# 03 · Training Orca (AlphaZero-style)

This notebook walks through the AlphaZero-style training pipeline that produces the **Orca** bot. By the end you will have run a short training session, watched the loss / ELO curves in TensorBoard, and compared two checkpoints head-to-head.

Estimated time: 20 minutes (or 5 if you skim the training step). Best run on a Colab T4 GPU. Flip **Runtime → Change runtime type → GPU** before starting.

## The shape of AlphaZero training

Each iteration of the main loop does three things:

1. **Self-play.** The current network plays games against itself. Every move runs MCTS guided by the policy / value heads; the search visit distribution and the eventual game outcome become a training sample.
2. **Train.** Stochastic gradient descent on the network: cross-entropy on the MCTS visit distribution (policy head) plus MSE on the game outcome (value head).
3. **Evaluate.** Optionally pit the new network against the previous best, and promote only if it wins more than half.

Repeat. ELO climbs, the network gets stronger, MCTS guided by the better network produces better data, and the loop closes.

See [Concepts](https://github.com/Saiki77/hexbot-building-framework/wiki/Concepts) for a deeper treatment of MCTS, policy + value heads, and ELO.

In [ ]:
# Install hexbot + tensorboard (for the metrics dashboard)
!pip install --quiet hexbot tensorboard

## Inspect the included Orca

The framework ships a pre-trained checkpoint (early stage, around iteration 65). It is enough to make Orca beat the heuristic bot consistently, but not strong yet. We will train past it shortly.

In [ ]:
from hexbot import Bot, Arena

orca = Bot.orca()
result = Arena(orca, Bot.heuristic(), num_games=4).play(verbose=False)
print(f"Orca vs heuristic (4 games): {result}")

## Watch Orca think

`mcts_search` runs the AlphaZero search and returns visit counts. The top moves are the ones MCTS explored the most, i.e. the ones it considered most promising. This is how Orca actually picks moves under the hood.

In [ ]:
from hexbot import HexGame, mcts_search

g = HexGame()
g.place(0, 0)
g.place(2, 0); g.place(2, -1)

result = mcts_search(g, sims=200)
print(f"best move: {result['best_move']}")
print("top moves (by MCTS visit count):")
for move, visits in result['top_moves'][:5]:
    bar = '#' * (visits // 4)
    print(f"  {move}  visits={visits:>3}  {bar}")

## Run a short training session

Use the `colab-t4` profile to get sensible defaults for a free T4 GPU (batch size, workers, sims, games per iteration). For first-time exploration, 20 iterations is enough to see loss come down and ELO start moving.

Add `--tensorboard` so we can inspect metrics in the next cell.

In [ ]:
!python -m orca.train --profile=colab-t4 --iterations 20 --tensorboard

## Inspect the run

The trainer wrote a TensorBoard event file plus a manifest JSON under `runs/<timestamp>/`. Load TensorBoard inline:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

The interesting scalars to watch:

- `loss/total`, `loss/policy`, `loss/value`: should decrease.
- `elo/current`: should climb.
- `time/iter_seconds`: tells you how long an iteration takes; useful when picking iteration budgets for longer runs.

Also worth a look:

In [ ]:
import json, glob, os

manifest_path = sorted(glob.glob('runs/*/manifest.json'))[-1]
print(f"Manifest: {manifest_path}\n")
with open(manifest_path) as f:
    manifest = json.load(f)
for key in ['run_id', 'hexbot_version', 'git_sha', 'hostname', 'device']:
    print(f"  {key:20s} {manifest.get(key, '?')}")
print('\n  config snapshot:')
for k, v in manifest['config'].items():
    print(f"    {k:24s} {v}")

## Compare your trained checkpoint vs the bundled one

In [ ]:
from hexbot import Bot, Arena

# Grab the latest local checkpoint your training run produced
ckpts = sorted(glob.glob('hex_checkpoint_*.pt'),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
latest = ckpts[-1]
print(f"latest checkpoint: {latest}")

# Inspect the embedded _hexbot_meta
import torch
meta = torch.load(latest, weights_only=False)['_hexbot_meta']
print(f"  arch={meta['arch']}  iter={meta['iter']}  elo={meta['elo']}")

trained = Bot.from_checkpoint(latest)
bundled = Bot.orca()
result = Arena(trained, bundled, num_games=4).play(verbose=False)
print(f"\nyour {meta['iter']}-iter bot vs bundled Orca: {result}")

## Preview AutoTuner decisions without applying them

`AutoTuner` is the rule-based hyperparameter controller that nudges learning rate, training steps, and game mix between iterations. To understand what it would do without committing to the changes, run training with `--auto-tuner-dry-run`.

In [ ]:
!python -m orca.train --profile=colab-t4 --iterations 3 --auto-tuner-dry-run 2>&1 | grep -i AutoTuner | head

## Share your bot (Model Zoo)

Package a checkpoint with metadata so others can `Zoo.download('your-name')`. Local-only example:

In [ ]:
from orca.zoo import Zoo

Zoo.package(
    latest, output_path='my-first-orca.pt',
    name='my-first-orca', author='you',
    elo=int(meta.get('elo') or 1000),
    description=f"Trained {meta['iter']} iterations in a notebook",
)
print('packaged: my-first-orca.pt and my-first-orca.json')
Zoo.list()

To publish for real, install the `gh` CLI, authenticate, and call `Zoo.upload('my-first-orca.pt', name='my-first-orca')`. The framework's [Featured Community Bots table](https://github.com/Saiki77/hexbot-building-framework#featured-community-bots) is auto-regenerated from the leaderboard, so once you're rated you show up there as well.

## Next steps

- Increase the iteration count. ELO milestones in the [FAQ](https://github.com/Saiki77/hexbot-building-framework/wiki/FAQ): ~30 iters beats heuristic, ~100 looks coherent, ~300 competitive.
- Try a different architecture: `--config hex-masked` or `--config large`.
- Try the SFT warm-start: scrape human games then fine-tune. See [SFT Guide](https://github.com/Saiki77/hexbot-building-framework/wiki/SFT-Guide).
- Run a hyperparameter sweep with `python -m orca.sweep --trials 20` (requires `pip install 'hexbot[sweep]'`).
- Play your trained bot live on hexo.did.science: see [Playing Online](https://github.com/Saiki77/hexbot-building-framework/wiki/Playing-Online).